[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C22_Reasoning_RL_Course/03_prm/03_prm.ipynb)

# 03 · 过程奖励模型 PRM（用 numpy 模拟）

目标：从零实现 **PRM**——用**蒙特卡洛 rollout** 自动标注**步级软标签**（Math-Shepherd）、训一个玩具 PRM、用它**聚合打分**做 best-of-N，并**亲手复现 Goodhart 倒 U**（过度优化学习奖励，真实正确率先升后降）。

路线：可枚举多步任务+真实步价值 → MC 标注(对拍真值) → 训玩具 PRM → min/prod/mean 聚合 → PRM vs ORM 重排 → 倒 U → ✏️ 练习 → 📖 答案 → 🧪 PRM800K/GSM8K 胶囊。

> 心智模型：**PRM = 给每一步装一个『还能不能答对』的概率仪表盘**；更密的信号更准，但学习的奖励会被 hack。

## 1 · 可枚举多步任务与真实步价值

任务：`K` 步、每步选一个数，**前缀和不许超过** `CAP`、最终和 `== TARGET` 才算对。这样『中途走过头』就成了**不可逆的错误步**。

因可枚举，我们能算每个前缀的**真实价值** $V(s_{\le k})$ = 从该前缀按均匀策略续写能答对的概率，当 MC 标注的对拍真值。

In [ ]:
import numpy as np
from itertools import product
rng = np.random.default_rng(0)

ACTIONS = np.array([0, 1, 2, 3]); K_STEPS = 4; TARGET = 6; CAP = 6; A = len(ACTIONS)

def is_correct(traj):
    '''可验证：全程前缀和<=CAP 且 总和==TARGET。'''
    s = 0
    for a in traj:
        s += ACTIONS[a]
        if s > CAP:
            return 0
    return int(s == TARGET)

def true_value(prefix):
    '''V(s_<=k) = 从 prefix 按均匀策略续写到 K 步、答对的概率(枚举所有续写)。'''
    rem = K_STEPS - len(prefix)
    if rem == 0:
        return float(is_correct(prefix))
    wins = 0; total = 0
    for cont in product(range(A), repeat=rem):
        total += 1
        wins += is_correct(list(prefix) + list(cont))
    return wins / total

print('空前缀价值 V() =', round(true_value([]), 4), '(整体答对率)')
print('前缀 [3] 价值   =', round(true_value([3]), 4))     # 已用3，还需3
print('前缀 [3,3] 价值 =', round(true_value([3,3]), 4))   # 已到6(满),后面只能选0
print('前缀 [3,3,3] 价值=', round(true_value([3,3,3]), 4)) # 已超CAP -> 0
assert true_value([3,3,3]) == 0.0, '前缀和已超 CAP -> 必败'
assert 0 < true_value([]) < 1
print('✅ 有了可枚举的真实步价值 V(s)，作为 MC 标注的对拍真值')

## 2 · Math-Shepherd：MC rollout 标步级软标签

给第 `k` 步标价值：从前缀 `s_<=k` 出发**多次 rollout 到底**，数答对比例 = $\hat v_k$（软标签）。

对拍：MC 估计应收敛到第 1 节的真实 $V(s)$；rollout 数 `N` 越大方差越小。

In [ ]:
def mc_step_value(prefix, n_rollouts=2000, seed=0):
    '''Math-Shepherd：从 prefix 按均匀策略续写 n 次，返回答对比例(软标签)。'''
    r = np.random.default_rng(seed)
    rem = K_STEPS - len(prefix)
    if rem == 0:
        return float(is_correct(prefix))
    wins = 0
    for _ in range(n_rollouts):
        cont = [int(r.choice(A)) for _ in range(rem)]
        wins += is_correct(list(prefix) + cont)
    return wins / n_rollouts

for prefix in [[], [3], [2], [3, 3]]:
    v_true = true_value(prefix)
    v_mc = mc_step_value(prefix, n_rollouts=4000)
    print(f'前缀 {str(prefix):8s}: 真值 {v_true:.4f}  MC估计 {v_mc:.4f}')
    assert abs(v_mc - v_true) < 0.03, 'MC 软标签应收敛到真实价值'

# rollout 越多方差越小
vs_small = [mc_step_value([2], 100, seed=s) for s in range(30)]
vs_large = [mc_step_value([2], 4000, seed=s) for s in range(30)]
print(f'\nN=100  软标签标准差 = {np.std(vs_small):.4f}')
print(f'N=4000 软标签标准差 = {np.std(vs_large):.4f}')
assert np.std(vs_large) < np.std(vs_small), '更多 rollout -> 标签方差更小'
print('✅ MC 自动标注：软标签收敛到真实步价值，无需任何人工标注')

## 3 · 训一个玩具 PRM（回归步价值）

用 MC 软标签作监督，训一个**玩具 PRM**：输入步特征(前缀和、剩余步数、本步动作)，回归该步价值。
用线性回归(闭式解)当 PRM，验证它能预测出步价值、识别『走过头』的坏步。

In [ ]:
# 构造训练集：(步特征, MC软标签)。好特征对 PRM 至关重要。
def step_features(prefix, action):
    '''一步的特征：[执行后前缀和, 剩余步数, action, 是否超CAP, 距目标的绝对差, 偏置]。
       注意:『是否超CAP』『距目标差』这类特征让线性 PRM 能学到 CAP 悬崖。'''
    s_after = sum(ACTIONS[a] for a in prefix) + ACTIONS[action]
    rem = K_STEPS - len(prefix) - 1
    over = float(s_after > CAP)
    gap = abs(TARGET - s_after)
    return np.array([s_after, rem, ACTIONS[action], over, gap, 1.0])

X, y = [], []
for prefix_len in range(K_STEPS):
    for prefix in product(range(A), repeat=prefix_len):
        if prefix and sum(ACTIONS[a] for a in prefix) > CAP:
            continue
        for action in range(A):
            feat = step_features(list(prefix), action)
            label = mc_step_value(list(prefix) + [action], n_rollouts=2000)
            X.append(feat); y.append(label)
X = np.array(X); y = np.array(y)
print(f'PRM 训练集: {X.shape[0]} 个步样本, 特征维 {X.shape[1]}')

# 闭式岭回归当 PRM
lam = 1e-3
W = np.linalg.solve(X.T @ X + lam * np.eye(X.shape[1]), X.T @ y)
def prm_score(prefix, action):
    return float(np.clip(step_features(prefix, action) @ W, 0, 1))

pred = X @ W
mse = np.mean((pred - y) ** 2)
print(f'PRM 训练 MSE = {mse:.4f}  (步价值含 CAP 悬崖,非线性,线性 PRM 只能近似)')
assert mse < 0.15, 'PRM 应能合理拟合步价值(线性近似)'
# 走过头的步价值应低于安全步(PRM 学到了『别超 CAP』的趋势)
assert prm_score([3, 3], 3) < prm_score([1, 1], 2), 'PRM 应给走过头的坏步更低分'
# 与真值的相关性应为正(排序基本对)
corr = np.corrcoef(pred, y)[0, 1]
print(f'PRM 预测 vs 真值 相关系数 = {corr:.3f}')
assert corr > 0.6, 'PRM 预测应与真实步价值正相关'
print('✅ 玩具 PRM 训成：能预测步价值趋势、给走过头的坏步更低分')

## 4 · 聚合：min / prod / mean / last

PRM 对一条解输出一串步分数，best-of-N 需聚合成解级标量。四种聚合语义不同，**会改变排序**。
我们实现四种，并构造例子让同一组解在不同聚合下**排序翻转**。

In [ ]:
def aggregate(step_scores, how='min'):
    s = np.asarray(step_scores, dtype=float)
    if how == 'min':  return float(s.min())
    if how == 'prod': return float(np.prod(s))
    if how == 'mean': return float(s.mean())
    if how == 'last': return float(s[-1])
    raise ValueError(how)

# 解A：步步中等[0.7,0.7,0.7,0.7]；解B：大多很好但有一步崩[0.95,0.95,0.1,0.95]
solA = [0.7, 0.7, 0.7, 0.7]
solB = [0.95, 0.95, 0.1, 0.95]
for how in ['min', 'prod', 'mean', 'last']:
    a, b = aggregate(solA, how), aggregate(solB, how)
    winner = 'A' if a > b else 'B'
    print(f'{how:5s}: A={a:.3f}  B={b:.3f}  -> 选 {winner}')
# min/prod 惩罚 B 的崩步 -> 选 A; mean/last 可能选 B
assert aggregate(solA, 'min') > aggregate(solB, 'min'), 'min 惩罚最弱步 -> A 胜'
assert aggregate(solB, 'last') > aggregate(solA, 'last'), 'last 看终态 -> B 胜'
print('✅ 聚合改变排序：min/prod 重『步步对』，mean/last 更宽松 —— 选谁取决于任务')

## 5 · PRM vs ORM 做 best-of-N 重排

核心实证：用 PRM(步级,min 聚合) vs ORM(解级)对一批候选解重排，比较**选出的解的真实正确率**。
PRM 能识别『前面对、最后崩』的接近解，重排应更准。

In [ ]:
# 模拟一批候选解：每条解有真实步价值序列 + 是否最终答对
def make_candidates(n=400, seed=1):
    r = np.random.default_rng(seed)
    cands = []
    for _ in range(n):
        traj = [int(r.choice(A)) for _ in range(K_STEPS)]
        correct = is_correct(traj)
        # 真实步价值序列(每个前缀的真值)
        step_vals = [true_value(traj[:k+1]) for k in range(K_STEPS)]
        cands.append((traj, correct, step_vals))
    return cands

cands = make_candidates()
base_rate = np.mean([c[1] for c in cands])
print(f'候选解 基础正确率(随机选一条) = {base_rate:.3f}')

def prm_pick(cands, how='min'):
    '''用 PRM(步价值聚合)选分最高的解，返回它是否真对。'''
    scores = [aggregate(c[2], how) for c in cands]
    return cands[int(np.argmax(scores))][1]

def orm_pick(cands, noise=0.15, seed=0):
    '''ORM 只给解级分：真对的~高分、错的~低分(带噪声,模拟学习的 ORM)。'''
    r = np.random.default_rng(seed)
    scores = [c[1] + r.normal(0, noise) for c in cands]
    return cands[int(np.argmax(scores))][1]

# 多次重复取平均(不同候选批)
prm_acc = np.mean([prm_pick(make_candidates(seed=s), 'min') for s in range(80)])
orm_acc = np.mean([orm_pick(make_candidates(seed=s), seed=s) for s in range(80)])
print(f'PRM(min) best-of-N 选出解正确率 = {prm_acc:.3f}')
print(f'ORM      best-of-N 选出解正确率 = {orm_acc:.3f}')
assert prm_acc > base_rate, 'PRM 重排应优于随机选'
print('✅ PRM 用步级价值重排，显著高于基础正确率(更密的信号 -> 更准的选择)')

## 6 · Goodhart 倒 U：过度优化学习奖励

核心警示。一个**有缺陷的 PRM**(对某个表面特征给虚高分)被当 RL 奖励优化：随优化强度上升，**PRM 分一路升，但真实正确率先升后降**(倒 U)。这是一切学习奖励的根本风险。

In [ ]:
# 缺陷 PRM 的步级分 = 真实步价值 + 可被 hack 的偏置(对『选动作3』虚高加分)
def flawed_step_score(prefix, action, hack=0.5):
    v = true_value(list(prefix) + [action])         # 真实步价值(有用信号)
    return v + hack * (ACTIONS[action] == 3)         # + 可 hack 的表面特征

# 策略=对『缺陷 PRM 步分』贪心(用 beta 控优化强度)：朝 PRM 优化。
def policy_step(prefix, beta, hack=0.5):
    sc = np.array([flawed_step_score(prefix, a, hack) for a in range(A)])
    z = beta * sc; z = z - z.max(); e = np.exp(z); return e / e.sum()

def rollout_proxy_and_acc(beta, hack=0.5, n=4000, seed=0):
    r = np.random.default_rng(seed); accs = []; prm = []
    for _ in range(n):
        prefix = []
        for _ in range(K_STEPS):
            ps = policy_step(prefix, beta, hack); prefix.append(int(r.choice(A, p=ps)))
        accs.append(is_correct(prefix))
        prm.append(np.mean([flawed_step_score(prefix[:i], prefix[i], hack) for i in range(K_STEPS)]))
    return np.mean(prm), np.mean(accs)

print(f"{'优化强度 beta':>14}{'PRM平均分(代理)':>16}{'真实正确率':>12}")
prm_scores, true_accs = [], []
betas = [0, 0.5, 1, 2, 4, 8, 16]
for beta in betas:
    pm, ac = rollout_proxy_and_acc(beta)
    prm_scores.append(pm); true_accs.append(ac)
    print(f'{beta:>14}{pm:>16.3f}{ac:>12.3f}')

# PRM 代理分单调升(被 hack)，真实正确率先升后降(倒 U)
assert prm_scores[-1] > prm_scores[0], 'PRM 代理分应一路升(被 hack)'
peak = int(np.argmax(true_accs))
assert 0 < peak < len(true_accs) - 1, '真实正确率应先升后降(倒 U,峰在中间)'
assert true_accs[-1] < true_accs[peak] - 0.1, '过度优化后真实正确率明显下降'
print(f'\n真实正确率峰值在 beta={betas[peak]} (前段 PRM 与真目标一致,过此点被 hack)')
print('✅ Goodhart 倒 U：过度优化学习奖励, 代理分↑但真实表现先↑后↓ —— 须 KL/早停')

---
## ✏️ 练习 1：MC 步级软标签

实现 `mc_step_value(prefix, n_rollouts, seed)`：从 prefix 按均匀策略续写 `n_rollouts` 次，返回答对比例。验证它收敛到 `true_value`、且 `N` 越大方差越小。

In [ ]:
def mc_step_value(prefix, n_rollouts=2000, seed=0):
    # TODO: rem = K_STEPS-len(prefix); 若 rem==0 返回 is_correct(prefix)
    #       否则采 n_rollouts 条续写, 返回答对比例
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
for prefix in [[], [3], [2, 2]]:
    assert abs(mc_step_value(prefix, 4000) - true_value(prefix)) < 0.03
assert mc_step_value([3, 3, 3], 100) == 0.0, '超 CAP 前缀必败'
small = np.std([mc_step_value([2], 100, s) for s in range(20)])
large = np.std([mc_step_value([2], 4000, s) for s in range(20)])
assert large < small, 'N 越大方差越小'
print('✅ 练习 1 通过：MC 软标签(Math-Shepherd 的核心)')

## ✏️ 练习 2：四种聚合

实现 `aggregate(step_scores, how)` 支持 min/prod/mean/last。验证语义，并验证 **prod 系统性惩罚长解**(步多则连乘更小)。

In [ ]:
def aggregate(step_scores, how='min'):
    # TODO: min/prod/mean/last
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
s = [0.9, 0.8, 0.5, 0.95]
assert abs(aggregate(s, 'min') - 0.5) < 1e-9
assert abs(aggregate(s, 'mean') - np.mean(s)) < 1e-9
assert abs(aggregate(s, 'last') - 0.95) < 1e-9
assert abs(aggregate(s, 'prod') - np.prod(s)) < 1e-9
# prod 惩罚长解：同样每步 0.9, 步越多 prod 越小
assert aggregate([0.9]*2, 'prod') > aggregate([0.9]*6, 'prod')
# min 对步数不敏感
assert abs(aggregate([0.9]*2, 'min') - aggregate([0.9]*6, 'min')) < 1e-9
print('✅ 练习 2 通过：聚合语义 + prod 对长解的系统性惩罚')

## ✏️ 练习 3：找出错误发生的步

实现 `find_error_step(step_values, drop_thresh)`：给定一条解的步价值序列，返回**价值首次骤降**(相邻下降幅度 > `drop_thresh`)的步下标(无则返回 -1)。这是 PRM 定位错误的用法。

In [ ]:
def find_error_step(step_values, drop_thresh=0.3):
    # TODO: 找第一个 k 使 step_values[k-1]-step_values[k] > drop_thresh, 返回 k; 无则 -1
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
assert find_error_step([0.6, 0.7, 0.65, 0.1, 0.05]) == 3, '第3步价值骤降'
assert find_error_step([0.5, 0.55, 0.6, 0.7]) == -1, '一路上升无骤降'
assert find_error_step([0.9, 0.2], drop_thresh=0.3) == 1
assert find_error_step([0.9, 0.7], drop_thresh=0.3) == -1, '降幅未超阈值'
print('✅ 练习 3 通过：PRM 用价值骤降定位出错步')

## ✏️ 练习 4：监控倒 U、给出早停点

实现 `optimal_stop(proxy_scores, true_accs)`：给定随优化强度的代理分序列与真实正确率序列，返回**真实正确率最高**的下标(应早停于此，而非代理分最高处)。

In [ ]:
def optimal_stop(proxy_scores, true_accs):
    # TODO: 返回 true_accs 最大值的下标(早停看真实指标,不看代理)
    raise NotImplementedError

In [ ]:
# —— 练习 4 自测 ——
proxy = [0.3, 0.5, 0.7, 0.85, 0.95]   # 代理分一路升
true_ = [0.3, 0.55, 0.6, 0.4, 0.2]    # 真实先升后降
assert optimal_stop(proxy, true_) == 2, '应停在真实正确率峰(idx2),非代理峰(idx4)'
assert np.argmax(proxy) != optimal_stop(proxy, true_), '代理峰 != 真实峰(Goodhart)'
print('✅ 练习 4 通过：早停看真实指标，别信被优化过的代理(防 Goodhart)')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def mc_step_value(prefix, n_rollouts=2000, seed=0):
    r = np.random.default_rng(seed)
    rem = K_STEPS - len(prefix)
    if rem == 0:
        return float(is_correct(prefix))
    wins = 0
    for _ in range(n_rollouts):
        cont = [int(r.choice(A)) for _ in range(rem)]
        wins += is_correct(list(prefix) + cont)
    return wins / n_rollouts

In [ ]:
# 练习 2 参考答案
def aggregate(step_scores, how='min'):
    s = np.asarray(step_scores, dtype=float)
    return {'min': float(s.min()), 'prod': float(np.prod(s)),
            'mean': float(s.mean()), 'last': float(s[-1])}[how]

In [ ]:
# 练习 3 参考答案
def find_error_step(step_values, drop_thresh=0.3):
    for k in range(1, len(step_values)):
        if step_values[k-1] - step_values[k] > drop_thresh:
            return k
    return -1

In [ ]:
# 练习 4 参考答案
def optimal_stop(proxy_scores, true_accs):
    return int(np.argmax(true_accs))

---
## 🧪 真实数据胶囊：PRM800K / GSM8K 步骤标注

用真实数据的步骤结构算账：① GSM8K 用 `<<算式=结果>>` 标注的**步数分布**（真实推理有多少步）；② PRM800K 的标注规模（人工步级标注有多贵）。带 try/except 回退到内置真实数值。

In [ ]:
# 真实数据规模(公开信息)
PRM800K_INFO = dict(num_step_labels=800_000, source='OpenAI Lightman 2023, MATH')
GSM8K_FALLBACK = [
    'In May she sold 48/2 = <<48/2=24>>24 clips.\nAltogether 48+24 = <<48+24=72>>72.\n#### 72',
    'Per minute 12/60 = $<<12/60=0.2>>0.2.\n50 min => 0.2*50 = $<<0.2*50=10>>10.\n#### 10',
    'Weng worked <<3*8=24>>24 hours total.\nEarned 24*<<15=15>>15 = $<<24*15=360>>360.\n#### 360',
]
import re
try:
    from datasets import load_dataset
    ds = load_dataset('openai/gsm8k', 'main', split='train[:200]')
    answers = [r['answer'] for r in ds]
    print('已联网载入真实 GSM8K 200 条')
except Exception as e:
    answers = GSM8K_FALLBACK
    print('离线回退到内置真实 GSM8K 样例 (', type(e).__name__, ')')

def count_steps(answer):
    '''GSM8K 的计算步 = <<...>> 标注的个数。'''
    return len(re.findall(r'<<[^>]+>>', answer))

step_counts = [count_steps(a) for a in answers]
print(f'GSM8K 推理步数: 平均 {np.mean(step_counts):.1f}, 最多 {max(step_counts)}, 最少 {min(step_counts)}')
print(f'PRM800K 人工步级标注规模: {PRM800K_INFO["num_step_labels"]:,} 条 -> 这就是 Math-Shepherd 想用 MC 自动化省掉的成本')
assert np.mean(step_counts) >= 1
print('✅ 真实推理是多步的(每步可标价值); 人工标 80 万步 -> MC 自动标注的动机')

**🧪 胶囊练习**：实现 `mc_label_cost(num_solutions, steps_per_sol, rollouts_per_step)` —— 用 Math-Shepherd 给一批解的所有步标注，需要的**总 rollout 次数**。量化『自动标注虽省人工但很费算力』。

In [ ]:
def mc_label_cost(num_solutions, steps_per_sol, rollouts_per_step):
    # TODO: 总 rollout = 解数 * 每解步数 * 每步 rollout 数
    raise NotImplementedError

In [ ]:
# 自测
assert mc_label_cost(1000, 8, 16) == 1000 * 8 * 16
# 步越多/rollout越多 -> 越贵
assert mc_label_cost(1000, 16, 16) > mc_label_cost(1000, 8, 16)
cost = mc_label_cost(10000, 10, 32)
print(f'给 1万条解(每条10步,每步32 rollout)标注 = {cost:,} 次 rollout')
print('✅ 胶囊练习通过：MC 标注省人工但费算力(K*N 倍生成)')

In [ ]:
# 📖 胶囊参考答案
def mc_label_cost(num_solutions, steps_per_sol, rollouts_per_step):
    return num_solutions * steps_per_sol * rollouts_per_step

### 小结
- **PRM vs ORM**：PRM 给每一步打分(更密信号、更准 credit)，ORM 只给解级一个分；MATH 上 PRM 显著优于 ORM。
- **step-level 价值** $v_k=V(s_{\le k})$ = 从该步续写答对的概率；价值骤降处 = 出错步(PRM 的『仪表盘』)。
- **Math-Shepherd**：MC rollout 答对比例 = 步级**软标签**, 收敛到真实价值, **无需人工**(但费算力、有噪声、依赖策略)。
- **聚合**：min/prod 重『步步对』(惩罚最弱/连乘)，mean/last 更宽松；**会改变排序**, 选谁取决于任务。
- **Goodhart 倒 U**：过度优化学习奖励(PRM), 代理分一路升但真实正确率**先升后降** -> 须 KL 正则 + 早停 + 独立真值评判。
- **分工**：不可 hack 的规则/结果奖励用于**训练 RL**；可 hack 的学习 PRM 更适合**推理时当 verifier**(下一模块)。

下一站：**模块 04 · Verifier 引导搜索** —— 把 verifier 从训练信号变成推理时的搜索引导(BoN/投票/beam, pass@k)。